# Module 5 — Inverse Kinematics
## Unit 3 — Analytical (Closed-Form) Inverse Kinematics
### Lesson 3.4 — Analytical Inverse Kinematics (Unit 3 Recap)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Unit 3 synthesis
Closed form + atan2 + decoupling, end to end.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def ik_2link_closed(x, y, L1, L2):
    """Closed-form: return both (theta1,theta2); [] unreachable; one on boundary."""
    c2 = (x*x + y*y - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        s2 = sign*np.sqrt(max(0.0, 1.0 - c2*c2))
        t2 = np.arctan2(s2, c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(s2) < 1e-6:
            break
    return sols

def jacobian_2link(theta1, theta2, L1, L2):
    s1, s12 = np.sin(theta1), np.sin(theta1+theta2)
    c1, c12 = np.cos(theta1), np.cos(theta1+theta2)
    return np.array([[-L1*s1 - L2*s12, -L2*s12],
                     [ L1*c1 + L2*c12,  L2*c12]])

L1,L2=0.4,0.3
# closed form, all quadrants, FK-verified
for tgt in [(0.6,0.0),(-0.4,0.2),(0.2,-0.5)]:
    sols=ik_2link_closed(*tgt,L1,L2)
    print(tgt,"->",len(sols),"solution(s)")
    for (t1,t2) in sols:
        assert np.allclose(fk_two_link(t1,t2,L1,L2),tgt,atol=1e-9)


## Check your work


In [ ]:
# both solutions everywhere reachable, none when not
assert len(ik_2link_closed(0.5,0.0,L1,L2))==2
assert ik_2link_closed(0.95,0.0,L1,L2)==[]
print("All checks passed.")
